# 07 — Instrumental Variables
**References:** Angrist & Pischke (2009) Ch. 4 · Imbens & Angrist (1994, *Econometrica*) · Angrist, Imbens & Rubin (1996, *JASA*) · Wright (1928) · MIT 14.387 · Stock, Wright & Yogo (2002)

## Narrative thread
```
Unobserved confounding -> The IV idea -> Wald & 2SLS -> LATE and compliers -> Weak instruments -> Judging instruments
```

## When adjustment is hopeless

Notebooks 05–06 assumed all confounders were measured. An **instrument** $Z$ identifies a
causal effect even with *unobserved* confounding, if it satisfies:

1. **Relevance:** $Z$ affects treatment $W$ ($\text{Cov}(Z, W) \neq 0$) — testable.
2. **Exclusion:** $Z$ affects $Y$ *only through* $W$ — untestable, the crux.
3. **Independence:** $Z$ is as-good-as-random w.r.t. the unobserved confounders.
4. **Monotonicity** (for LATE): no defiers — $Z$ never pushes anyone *away* from treatment.

DAG: $Z \to W \to Y$ with $U \to W$, $U \to Y$, and **no** arrow $Z \to Y$ or $U \to Z$.

Classic instruments: draft lottery for military service (Angrist 1990), quarter of birth
for schooling (Angrist & Krueger 1991), rainfall for economic shocks, judge leniency,
distance to college, and randomized *encouragement* in trials with noncompliance.

## The Wald estimator and 2SLS

With a binary instrument:
$$\hat\tau_{Wald} = \frac{E[Y \mid Z=1] - E[Y \mid Z=0]}{E[W \mid Z=1] - E[W \mid Z=0]}
= \frac{\text{reduced form (ITT)}}{\text{first stage}}$$

**Two-stage least squares** generalizes: (1) regress $W$ on $Z$ (and exogenous controls),
(2) regress $Y$ on the fitted $\hat W$. Equivalently
$\hat\beta_{2SLS} = (Z^\top X)^{-1} Z^\top y$ in the just-identified case.
*Never* run the two stages manually for inference — the second-stage OLS standard errors
are wrong (they ignore first-stage noise).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [ ]:
# ── IV vs OLS under unobserved confounding ────────────────────────────────
# Returns to schooling with unobserved "ability":
#   ability -> schooling, ability -> wages   (confounder, unobserved)
#   Z = distance-to-college encouragement -> schooling only (instrument)
n = 50_000
ability = np.random.normal(0, 1, n)                # UNOBSERVED
z = np.random.binomial(1, 0.5, n)                  # instrument (as-if random)
schooling = 12 + 1.0*z + 1.5*ability + np.random.normal(0, 1, n)
wage      = 20 + 2.0*schooling + 4.0*ability + np.random.normal(0, 2, n)   # true effect = 2

df = pd.DataFrame(dict(z=z, w=schooling, y=wage))

ols = smf.ols('y ~ w', df).fit().params['w']

# 2SLS in one shot: beta = (Z'X)^(-1) Z'y  (just-identified, with intercepts)
Zm = np.column_stack([np.ones(n), z])
Xm = np.column_stack([np.ones(n), schooling])
beta_2sls = np.linalg.solve(Zm.T @ Xm, Zm.T @ df.y.values)

# Correct 2SLS standard errors: resid uses ACTUAL W, bread uses fitted W
resid = df.y.values - Xm @ beta_2sls
Xhat  = Zm @ np.linalg.lstsq(Zm, Xm, rcond=None)[0]
cov   = resid.var(ddof=2) * np.linalg.inv(Xhat.T @ Xhat)
se_2sls = np.sqrt(np.diag(cov))[1]

wald = (df.query('z==1').y.mean() - df.query('z==0').y.mean()) / \
       (df.query('z==1').w.mean() - df.query('z==0').w.mean())

print(f'True causal effect of schooling: 2.000')
print(f'OLS  (confounded by ability):    {ols:.3f}')
print(f'Wald estimator:                  {wald:.3f}')
print(f'2SLS estimate (SE):              {beta_2sls[1]:.3f} ({se_2sls:.3f})')

## What does IV estimate? The LATE theorem

With heterogeneous effects, IV does **not** recover the ATE. Classify units by how
treatment responds to the instrument:

| Type | $W(Z{=}0)$ | $W(Z{=}1)$ |
|---|---|---|
| Never-taker | 0 | 0 |
| **Complier** | 0 | 1 |
| Always-taker | 1 | 1 |
| Defier (ruled out) | 1 | 0 |

**LATE theorem** (Imbens & Angrist 1994): under assumptions 1–4,
$$\hat\tau_{IV} \xrightarrow{p} E[Y(1) - Y(0) \mid \text{complier}]$$

IV estimates the effect **for compliers only** — the units whose treatment the instrument
moves. Different instruments → different complier populations → legitimately different
estimates for the same treatment ("external validity is instrument-specific").


In [ ]:
# ── LATE by simulation: IV recovers the complier effect, not the ATE ─────
n = 200_000
u = np.random.normal(0, 1, n)                      # unobserved confounder
z = np.random.binomial(1, 0.5, n)

# Compliance types: 20% always, 40% compliers, 40% never
types = np.random.choice(['always', 'complier', 'never'], n, p=[0.2, 0.4, 0.4])
w = np.where(types == 'always', 1, np.where(types == 'complier', z, 0))

# Heterogeneous effects BY TYPE: always 8, compliers 4, never 1
tau = np.select([types=='always', types=='complier', types=='never'], [8.0, 4.0, 1.0])
y0 = 10 + 3*u + np.random.normal(0, 1, n)          # confounded baseline
y  = y0 + tau*w                                     # observed outcome

ate      = tau.mean()
late_true = tau[types == 'complier'].mean()
itt   = y[z==1].mean() - y[z==0].mean()
first = w[z==1].mean() - w[z==0].mean()
iv    = itt / first

print(f'ATE (all units):            {ate:.3f}')
print(f'True complier effect:       {late_true:.3f}')
print(f'First stage (share compliers): {first:.3f}')
print(f'ITT (reduced form):         {itt:.3f}')
print(f'IV = ITT / first stage:     {iv:.3f}   -> recovers the LATE, not the ATE')

## Weak instruments

If the first stage is weak, 2SLS is biased toward OLS (in finite samples) and wildly
unstable. Rules of thumb and remedies:

- **First-stage F-statistic > 10** (Staiger & Stock 1997); modern work asks for far more
  (Lee et al. 2022 suggest F > 100 for reliable 5% tests).
- Report the reduced form (ITT) — it is unbiased regardless of instrument strength.
- Weak-instrument-robust inference: Anderson-Rubin test.


In [ ]:
# ── Weak instrument pathology: sampling distribution of 2SLS ─────────────
def one_iv_draw(gamma, n=500):
    """gamma = first-stage strength. True effect = 1."""
    u = np.random.normal(0, 1, n)
    z = np.random.normal(0, 1, n)
    w = gamma*z + u + np.random.normal(0, 1, n)
    y = 1.0*w + 2*u + np.random.normal(0, 1, n)
    zc, wc, yc = z - z.mean(), w - w.mean(), y - y.mean()
    beta_iv  = (zc @ yc) / (zc @ wc)
    # first-stage F
    r = wc - zc * (zc @ wc) / (zc @ zc)
    F = ((zc @ wc)**2 / (zc @ zc)) / (r @ r / (n - 2))
    return beta_iv, F

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=False)
for ax, gamma in zip(axes, [1.0, 0.2, 0.05]):
    draws = np.array([one_iv_draw(gamma) for _ in range(4000)])
    b, F = draws[:, 0], draws[:, 1]
    b_clip = b[(b > -4) & (b < 6)]
    ax.hist(b_clip, bins=60, color='#1f77b4', alpha=0.85)
    ax.axvline(1.0, color='#d62728', lw=2)
    ax.set_title(f'first-stage F ~ {np.median(F):.0f}\nmedian 2SLS = {np.median(b):.2f}')
    ax.set_xlabel(r'$\hat\beta_{2SLS}$')
axes[0].set_ylabel('count')
plt.suptitle('2SLS sampling distribution as the instrument weakens (true effect = 1)', y=1.04)
plt.tight_layout(); plt.show()

## Key takeaways

| Concept | Statement |
|---|---|
| IV conditions | Relevance (testable) + exclusion + independence (arguments, not tests) |
| Wald / 2SLS | Causal effect = reduced form ÷ first stage |
| Manual 2SLS SEs | Wrong — use the proper 2SLS covariance (or a library) |
| LATE | IV identifies the complier-average effect, not the ATE |
| Monotonicity | "No defiers" — needed for the LATE interpretation |
| Weak instruments | F ≫ 10 required; report the ITT; use robust (AR) inference |
